# Bias-Variance Trade-Off



The bias-variance trade-off is a central problem in supervised learning. The **expected** error of a model on new data, averaged over many possible training sets, consists of 3 parts:

$MSE = variance + bias^2 + noise$ 

The noise is also called the **irreducible error**: it cannot be reduced, whichever model you choose.

The variance and the bias terms depend on the flexibility of the chosen model.


<img src="assets/bias_variance_total_error.svg" width="600">

*Image: [Bigbossfarin](https://commons.wikimedia.org/wiki/File:Bias_and_variance_contributing_to_total_error.svg), public domain (CC0).*

Normally, our goal is to minimise the total error (for example the mean squared error). Therefore, we would like to minimise both the bias and the variance terms of the error.

Unfortunately, it is typically impossible to do both simultaneously. **High-variance learning methods** may be able to represent their training set well but are at risk of **overfitting** to noisy or unrepresentative training data. In contrast, algorithms with **high bias** typically produce simpler models that may fail to capture important regularities (i.e. **underfit**) in the data.

This is called the **bias-variance trade-off**. It means that, if our goal is to minimise the squared error, it can be wise to use a biased estimator, as long as it reduces the variance.

## Learning Objectives

At the end of this notebook, you should be able to:

- Split data into a training and a test set with `train_test_split`, and calculate the RMSE on each.
- Train linear models with polynomial features of increasing degree.
- Compare training and test RMSE to identify underfitting (high bias) and overfitting (high variance).
- Compare models trained on different splits to show that flexible models have a higher variance.


## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import to divide our data into train and test data
from sklearn.model_selection import train_test_split

# import to create polynomial features
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# import of the linear regression model
from sklearn.linear_model import LinearRegression

# import of our evaluation metrics
from sklearn import set_config

# Make every scikit-learn transformer return a pandas DataFrame instead of a NumPy array
set_config(transform_output="pandas")

from sklearn.metrics import (
    r2_score,
    root_mean_squared_error,
)

RSEED = 12

In [ ]:
# Set constants and plotting parameters
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["figure.titlesize"] = 18
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (8, 6)

## Defining Our Data




In practice, we do not know the true relationship between the independent features (`X`) and the dependent variable (`y`). To understand the bias-variance trade-off, however, it helps to know it.

Here, we do know this relationship, because we define it ourselves in the **data-generating process**.

We choose `x` in the range from -7 to 7, and the true relationship between `x` and `y` is:


$y = 10x - 2 x^2 + 0.5 x^3 + \epsilon$

The term $\epsilon$ adds random, unpredictable noise to the data.



In [ ]:
# generate random data
np.random.seed(4)
# generating 100 random x-values between -7 and 7
x = np.random.uniform(-7, 7, size=100)

# generating corresponding y variable according to our defined true relationship
# adding some random noise with np.random.normal with mu = 0 and std = 20
y = 10 * x - 2 * (x**2) + 0.5 * (x**3) + np.random.normal(0, 20, 100)

# NumPy only generates the random numbers; we keep the data in pandas.
# scikit-learn expects the features as a table (a DataFrame) and the target as a Series.
x = pd.DataFrame({"x": x})
y = pd.Series(y, name="y")

## Train-Test Split



In most data science projects, one goal is to train a model that makes predictions for new instances. Training a model alone does not guarantee that it will perform well on unseen data. Before we can "trust" a model and put it into production, we need to evaluate its performance on data it has **not** been trained on.

The simplest way to do this is to split the original dataset into a **training set** and a **test set**:

- The training set is used to fit (train) the model.
- The test set contains only instances the model has never seen, so we use it to evaluate the model.

We do not have to split the dataset manually: scikit-learn's [`train_test_split()` function](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). 

In [ ]:
# Split data into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.8, random_state=RSEED
)  # Careful here: this is an unusual choice for the test_size. See the paragraph below for more explanations.

In [ ]:
# Size of train and test set
print("X_train: ", X_train.shape)
print("y_train: ", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

In the cells above, we called `train_test_split()` on our feature(s) and target. With `test_size`, we defined which share of the entire dataset ends up in the test set.

Another important parameter is `random_state`. Without it, scikit-learn **randomly** shuffles the data each time you call the function before it assigns instances to the test set. As a result, the scores of your model change from one run to the next. To reproduce the same results each time you run a notebook or script, you need to control this shuffling, and you do this by setting `random_state`.

**Exercise**:

The documentation of scikit-learn is very valuable. Have a look at the scikit-learn page for `train_test_split`. What is the default value of `test_size`?

<details><summary>Show answer</summary>
The default value for `test_size` is `None`, which means that the function will use a test size of 0.25 (25% of the data) if the `train_size` parameter is also `None`. If you specify a value for `train_size`, then the remaining data will be used for the test set.
</details>


The default values of scikit-learn are always a good starting point and a benchmark. So why do we deviate so much from the default here?

There are two competing concerns:

- With less training data, your parameter estimates have a greater variance.
- With less test data, your performance metric has a greater variance.

Broadly speaking, you should split the data so that neither variance is too high. This depends more on the absolute number of instances in each set than on the percentage.

Here, we want to show what a "greater variance" of the estimated parameters means, so our split is deliberately unconventional.


Let's have a look at our data. It is always good to get a feel for the training and test data, and to check whether they come from the same distribution or differ a lot.

In [ ]:
x_plot = pd.Series(np.linspace(-7, 7, 500), name="x")
y_plot_true = 10 * x_plot - 2 * (x_plot**2) + 0.5 * (x_plot**3)


def plot_models(lines, title, points="train_test"):
    """Plot the data, the true relationship and one curve per model.

    Args:
        lines (dict): maps a label to the predicted y values along x_plot.
        title (str): the title of the plot.
        points (str): which points to show: "train_test", "train" or "all".
    """
    if points == "all":
        plt.scatter(x, y, s=10, c="#193251", alpha=0.7, label="All data")
    else:
        plt.scatter(X_train, y_train, s=10, c="#193251", alpha=0.7, label="Train data")
        if points == "train_test":
            plt.scatter(X_test, y_test, s=10, c="#FF5A36", alpha=0.7, label="Test data")
    plt.plot(x_plot, y_plot_true, c="#5E5E5E", label='"True" relationship')
    for label, line in lines.items():
        plt.plot(x_plot, line, label=label)
    plt.legend()
    plt.xlabel("feature x")
    plt.ylabel("target variable y")
    plt.title(title)
    plt.show()

In [ ]:
plot_models({}, "Generated Data")

The true relationship is exactly our data generating process:


$y = 10x - 2 x^2 + 0.5 x^3 + \epsilon$


The data points scatter around this line because of the random error we added.

Overall, the training data covers nearly the same range of the feature `x` as the test data. Only in the far negative range (< -6) are there no training data.

## Modelling


We will try out different models on the generated data. Because the models differ in flexibility, we will also see how bias and variance affect them.



### First Model: Linear Regression



After defining our features and target and splitting the data into train and test sets, we can start modelling. As a first step, we create an instance of [`LinearRegression()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) and fit it on the **training data** with `.fit()`. Then we make predictions for both **train** and **test** sets using `.predict()`.

> Note: scikit-learn provides [`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html) (since v1.4) to compute RMSE directly. This is the recommended way in newer versions (e.g., v1.7.2) instead of using `mean_squared_error(..., squared=False)`.

We will report **R²** and **RMSE** for both train and test to evaluate model performance.

In [ ]:
# Instantiate our model
lin_reg = LinearRegression()

# Fit the model using our train data
lin_reg.fit(X_train, y_train)

# Make predictions on both train and test
y_pred_train = lin_reg.predict(X_train)
y_pred_test = lin_reg.predict(X_test)

So far, we have trained a model and made predictions on the training and the test set. Now it is time to evaluate the model's performance. We print the RMSE, the R-squared and the adjusted R-squared.

We use `root_mean_squared_error` from `sklearn.metrics` to calculate the RMSE. If you use `mean_squared_error` instead, take the square root of the result (for example with `np.sqrt()`) to get the RMSE.

Documentation: [`r2_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).

In [ ]:
# Let's define a function to calculate the R-squared adjusted 
def r2_score_adjusted(y_true, y_pred, n, p):
    """Calculate the adjusted R-squared value.

    Args:
        y_true (array-like): True target values.
        y_pred (array-like): Predicted target values.
        n (int): Number of observations.
        p (int): Number of features used in the model.

    Returns:
        float: Adjusted R-squared value
    """
    r2 = r2_score(y_true, y_pred)
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)


In [ ]:
# Calculate metrics
def calculate_metrics(name, y_train, y_pred_train, y_test, y_pred_test, p):
    """Print RMSE, R2 and adjusted R2 on the training and the test data.

    Args:
        name (str): the name of the model, printed as a header
        y_train (array): true values of y_train
        y_pred_train (array): predicted values of model for y_train
        y_test (array): true values of y_test
        y_pred_test (array): predicted values of model for y_test
        p (int): number of features used in the model
    """
    print(f"{name}:")
    print("---" * 10)
    for label, y_true, y_pred in [
        ("training", y_train, y_pred_train),
        ("test", y_test, y_pred_test),
    ]:
        print(f"Metrics on {label} data")
        print("RMSE:", round(root_mean_squared_error(y_true, y_pred), 3))
        print("R2:", round(r2_score(y_true, y_pred), 3))
        print("R2 Adjusted:", round(r2_score_adjusted(y_true, y_pred, len(y_true), p), 3))
        print("---" * 10)

In [ ]:
calculate_metrics(
    "Linear Regression Model", y_train, y_pred_train, y_test, y_pred_test, p=X_train.shape[1]
)

print("Standard deviation of y:", y.std())
print("Mean of y:", y.mean())

The optimal RMSE would be 0, which would mean that all predictions match the true values exactly.

Here, the RMSE on the training data (33) is smaller than the RMSE on the test data (44). This is typical: because we calculated the coefficients from the training data, the model tends to fit the training data better than unseen test data.

Let's look at our model graphically.

In [ ]:
line_lin_reg = lin_reg.intercept_ + lin_reg.coef_[0] * x_plot

In [ ]:
plot_models({"Linear model": line_lin_reg}, "Model Comparison")

The straight line is unable to capture the patterns in the data. This is an example of **underfitting**: the linear model is not flexible enough to capture the true relationship between `x` and `y`. We call such a model **biased**. It makes a strong assumption about the relationship between `x` and `y`, namely that it is linear.

With only one feature, we can see how the model performs by looking at the feature-target plot. With multiple features, this plot is no longer available. To evaluate the model anyway, we can plot the true `y` values against the predicted values, and plot the residuals. These plots show whether the residuals are randomly and normally distributed (the ideal situation) or whether they follow a pattern (which means the model can still be improved).

In [ ]:
def error_analysis(y_test, y_pred_test):
    """Generated true vs. predicted values and residual scatter plot for models

    Args:
        y_test (array): true values for y_test
        y_pred_test (array): predicted values of model for y_test
    """
    # Calculate residuals
    residuals = y_test - y_pred_test

    # Plot real vs. predicted values
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    plt.subplots_adjust(right=1)
    plt.suptitle("Error Analysis")

    ax[0].scatter(y_pred_test, y_test, color="#FF5A36", alpha=0.7)
    ax[0].plot([-400, 350], [-400, 350], color="#193251")
    ax[0].set_title("True vs. predicted values", fontsize=16)
    ax[0].set_xlabel("predicted values")
    ax[0].set_ylabel("true values")
    ax[0].set_xlim((y_pred_test.min() - 10), (y_pred_test.max() + 10))
    ax[0].set_ylim((y_test.min() - 40), (y_test.max() + 40))

    ax[1].scatter(y_pred_test, residuals, color="#FF5A36", alpha=0.7)
    ax[1].plot([-400, 350], [0, 0], color="#193251")
    ax[1].set_title("Residual Scatter Plot", fontsize=16)
    ax[1].set_xlabel("predicted values")
    ax[1].set_ylabel("residuals")
    ax[1].set_xlim((y_pred_test.min() - 10), (y_pred_test.max() + 10))
    ax[1].set_ylim((residuals.min() - 10), (residuals.max() + 10))

In [ ]:
error_analysis(y_test, y_pred_test)

Since

Residual = Observed – Predicted

positive residuals (on the y-axis) mean the prediction was too low, negative residuals mean the prediction was too high, and 0 means the prediction was exactly correct.

Ideally, the residuals:
- (1) are symmetrically distributed and cluster towards the middle of the plot.
- (2) are small compared with the spread of the target. Here the noise alone has a standard deviation of 20, so even a perfect model leaves residuals of about that size.
- (3) show no clear pattern.

Here, we still see patterns that the model is not capturing. For predicted values below -150, we always overestimate the y value; between -150 and about 0, we always underestimate it; and for predicted values above 0, the target value is again generally overestimated.

This again shows the underfitting of our highly biased model.



### Second Model: Linear Regression with Polynomial Features



To overcome underfitting, we need to increase the complexity of the model. We can do this by adding powers of the original feature as new features, which gives a higher-order equation.

The linear model


$\hat{y} = b_0 + b_1x $


can be transformed to

$\hat{y} = b_0 + b_1x + b_2x^2$


This is still considered a linear model, because it is linear in the **coefficients/weights** associated with the features.

**x²** is only a feature. 

However, the curve that we fit is quadratic. To convert the original features into their higher-order terms, we use scikit-learn's `PolynomialFeatures` class.

Documentation: [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

In [ ]:
# here we will add the second degree polynomial features
polynomial_features = PolynomialFeatures(degree=2, include_bias=False)
x_poly_train = polynomial_features.fit_transform(X_train)
x_poly_test = polynomial_features.transform(X_test)

model_poly = LinearRegression()
model_poly.fit(x_poly_train, y_train)
y_pred_test = model_poly.predict(x_poly_test)
y_pred_train = model_poly.predict(x_poly_train)

In [ ]:
calculate_metrics(
    "Polynomial Regression Model (degree 2)",
    y_train, y_pred_train, y_test, y_pred_test,
    p=x_poly_train.shape[1],
)

Adding quadratic features reduces the RMSE on both the training and the test data.

In [ ]:
# Predict along the plotting grid, transformed the same way as the training data
line_poly_2_reg = model_poly.predict(polynomial_features.transform(x_plot.to_frame()))

In [ ]:
plot_models(
    {"Linear model": line_lin_reg, "Degree 2 model": line_poly_2_reg},
    "Model Comparison",
)

The plot above shows the linear regression model fitted on the transformed features. The quadratic curve fits the data better than the straight line. Comparing the RMSE and the R² score of the two models also shows the improvement:

| Model (test set)   | RMSE  | R²  | R² Adjusted |
|---|---|---|---|
| **Linear Model**   | 44       |           0.79 |  0.78 |
| **Polynomial Model (Degree 2)**     |34     |            0.87 | 0.87 |


 


> The RMSE has decreased and the R² score has increased compared with the straight line.

Let's have a look at the residual plots. Do you think the points are randomly distributed, or can you still see a pattern?

In [ ]:
error_analysis(y_test, y_pred_test)

Even though the residuals got a bit smaller (the linear regression had residuals from -100 to 60), we can still see clear patterns.

Adding higher polynomial degrees might help to capture this pattern.

### Try It: Predict Before You Run

The next model adds cubic features (degree 3). Before you run it, write down your prediction: will the test RMSE fall or rise compared with the degree-2 model (33.82), and why? Look back at how the data was generated.

*Your prediction:*

<details><summary>Show answer</summary>

The test RMSE should **fall**, and it does: from 33.82 to 19.34. The data was generated from a cubic relationship, $y = 10x - 2x^2 + 0.5x^3 + \epsilon$, so the degree-2 model is still too simple to follow it: it has **bias**. A degree-3 model can represent the true relationship exactly, so most of that bias disappears. What is left is mainly the noise $\epsilon$, which no model can remove.

</details>

### Third Model: Degree-3 Polynomials

Let's add more flexibility: a cubic curve, with polynomial features up to degree 3.

Complete the next cell yourself. Follow the degree-2 model above: the steps are the same, only the degree changes.

In [ ]:
# TODO: create cubic features (degree=3) as for the degree-2 model above:
#       fit them on the training data, then transform the test data
polynomial_features = ...
x_poly_train = ...
x_poly_test = ...

# TODO: train a LinearRegression on the cubic features, then predict on both sets
model_3_poly = ...
y_pred_test = ...
y_pred_train = ...

In [ ]:
calculate_metrics(
    "Polynomial Regression Model (degree 3)",
    y_train, y_pred_train, y_test, y_pred_test,
    p=x_poly_train.shape[1],
)

In [ ]:
line_poly_3_reg = model_3_poly.predict(polynomial_features.transform(x_plot.to_frame()))

In [ ]:
plot_models(
    {
        "Linear model": line_lin_reg,
        "Degree 2 model": line_poly_2_reg,
        "Degree 3 model": line_poly_3_reg,
    },
    "Model Comparison",
)

In this plot, the linear model with cubic features is almost identical to the data-generating process.
The RMSE on the training and the test data is also low compared with the other models.


| Model (test set)   | RMSE  | R²  | R² Adjusted |
|---|---|---|---|
| **Linear Model**   | 44       |           0.79 |  0.78 |
| **Polynomial Model (Degree 2)**     |34     |            0.87 | 0.87 |
| **Polynomial Model (Degree 3)**     |19     |            0.96 | 0.957 |

In [ ]:
error_analysis(y_test, y_pred_test)

In the residual plots, the range of the residuals is much smaller, and there is no general pattern visible. This suggests that the model captures the relationship between the feature and the target variable that is present in the data.


### Fourth Model: Degree-8 Polynomials

Let's try an even more flexible model. This model should also be able to capture the patterns in the data.

Documentation: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
# Degree-8 features reach values in the millions (x**8), which makes the fit
# numerically inaccurate. Standardising them keeps it accurate; notebook 02
# explains scaling in detail.
polynomial_features = PolynomialFeatures(degree=8, include_bias=False)
scaler_8 = StandardScaler()
x_poly_train = scaler_8.fit_transform(polynomial_features.fit_transform(X_train))
x_poly_test = scaler_8.transform(polynomial_features.transform(X_test))

model_8_poly = LinearRegression()
model_8_poly.fit(x_poly_train, y_train)
y_pred_test = model_8_poly.predict(x_poly_test)
y_pred_train = model_8_poly.predict(x_poly_train)

In [ ]:
calculate_metrics(
    "Polynomial Regression Model (degree 8)",
    y_train, y_pred_train, y_test, y_pred_test,
    p=x_poly_train.shape[1],
)

This is a typical sign of **overfitting**. The degree-8 model has the lowest training RMSE of all the models so far (13.80), but its test RMSE rose to 32.92, clearly worse than the degree-3 model (19.34).
Overfitting happens when the model learns some of the residual variation (i.e., the noise) as if that variation represented the underlying relationship. This model will not give good results, because the irrelevant information (noise) it has learned does not help to predict `y` for unseen data.

| Model (test set)   | RMSE  | R²  | R² Adjusted |
|---|---|---|---|
| **Linear Model**   | 44       |           0.79 |  0.78 |
| **Polynomial Model (Degree 2)**     |34     |            0.87 | 0.87 |
| **Polynomial Model (Degree 3)**     |19     |            0.96 | 0.957 |
| **Polynomial Model (Degree 8)**     |33     |            0.88 | 0.87 |


In [ ]:
line_poly_8_reg = model_8_poly.predict(
    scaler_8.transform(polynomial_features.transform(x_plot.to_frame()))
)

In [ ]:
plot_models({"Degree 8 model": line_poly_8_reg}, "Model Comparison")

This model is much more flexible than the models we have used so far: it bends more to follow the training data. The model with polynomial features up to degree 8 is further away from the data-generating process than the previous model.

This curve may fit the training data very well, but it no longer fits the unseen test data so well.

Let's look at the training data only, for three of the models.

In [ ]:
# Training data only
plot_models(
    {
        "Linear model": line_lin_reg,
        "Degree 3 model": line_poly_3_reg,
        "Degree 8 model": line_poly_8_reg,
    },
    "Model Comparison",
    points="train",
)

With increasing flexibility, the models fit the training data better. The linear model with polynomials of order 1 clearly fails to capture the variation in the data. The linear model with polynomials up to order 8 fits the training data very well, but it may struggle on new instances (poor **generalisation**).



In [ ]:
error_analysis(y_test, y_pred_test)

For most points, the residual plot of this model looks acceptable. However, for two instances the model's predictions are very poor.



### Try It: Less Noise

Overfitting means fitting the **noise** in the training data. So what happens with less noise? Generate a second dataset in the same way, with `np.random.seed(4)` again, but with a noise standard deviation of **5** instead of 20. Split it with the same `test_size` and `random_state`, and train a degree-3 and a degree-8 model on it (standardise the degree-8 features as above). Compare the gap between the training and the test RMSE with the one you found for degree 8 (13.80 and 32.92).

In [ ]:
# Generate the data again with the same seed, but less noise
np.random.seed(4)
x_low = np.random.uniform(-7, 7, size=100)
# TODO: the same formula as the original data, with np.random.normal(0, 5, 100)
y_low = ...
x_low, y_low = pd.DataFrame({"x": x_low}), pd.Series(y_low, name="y")

# TODO: split with the same test_size and random_state as the original data
X_train_low, X_test_low, y_train_low, y_test_low = ...

for degree in [3, 8]:
    features = PolynomialFeatures(degree=degree, include_bias=False)
    scaler = StandardScaler()
    # TODO: fit_transform the training data, then only transform the test data
    train = ...
    test = ...
    # TODO: fit a LinearRegression on train, then calculate both RMSEs
    model = ...
    rmse_train = ...
    rmse_test = ...
    print(f"degree {degree}: train RMSE {rmse_train:.2f}, test RMSE {rmse_test:.2f}")

<details><summary>Show answer</summary>

```python
np.random.seed(4)
x_low = np.random.uniform(-7, 7, size=100)
y_low = 10 * x_low - 2 * (x_low**2) + 0.5 * (x_low**3) + np.random.normal(0, 5, 100)
x_low, y_low = pd.DataFrame({"x": x_low}), pd.Series(y_low, name="y")

X_train_low, X_test_low, y_train_low, y_test_low = train_test_split(
    x_low, y_low, test_size=0.8, random_state=RSEED
)

for degree in [3, 8]:
    features = PolynomialFeatures(degree=degree, include_bias=False)
    scaler = StandardScaler()
    train = scaler.fit_transform(features.fit_transform(X_train_low))
    test = scaler.transform(features.transform(X_test_low))
    model = LinearRegression().fit(train, y_train_low)
    print(
        f"degree {degree}: train RMSE {root_mean_squared_error(y_train_low, model.predict(train)):.2f}, "
        f"test RMSE {root_mean_squared_error(y_test_low, model.predict(test)):.2f}"
    )
```

With the same seed, the points and the noise follow the same pattern, only the noise is four times smaller. The degree-8 model now scores a training RMSE of 3.45 and a test RMSE of 8.23: a gap of 4.78, down from 19.11 with the original noise. With less noise there is less to memorise, so overfitting shrinks.

It does not disappear, though. The degree-3 model still does better on the test set (4.83, with a gap of only 0.92), because the extra flexibility of degree 8 still bends the curve to follow the remaining noise.

</details>

## "High Variance" Models



Let's have one more look at what "high-variance" models mean.

**Variance** is the amount by which the estimate of the target changes if different training data were used. In other words: how sensitive is the algorithm to the specific training data?

To see what this means in our case, we will train 3 inflexible models (polynomial degree 1) and 3 highly flexible models (polynomial degree 8) on different training data, and compare how much these models differ from one another.

The first step is to create 2 more train-test splits, which give us 2 more training sets to fit our models on.

In [ ]:
X_train_split2, X_test_split2, y_train_split2, y_test_split2 = train_test_split(
    x, y, test_size=0.8, random_state=91
)
X_train_split3, X_test_split3, y_train_split3, y_test_split3 = train_test_split(
    x, y, test_size=0.8, random_state=222
)

First, we use the original training set and the two new ones to train 3 linear models:

In [ ]:
# Model 1
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Model 2
lin_reg_2 = LinearRegression()
lin_reg_2.fit(X_train_split2, y_train_split2)

# Model 3
lin_reg_3 = LinearRegression()
lin_reg_3.fit(X_train_split3, y_train_split3)

Let's plot these three models to see how much they differ from one another.

In [ ]:
line_lin_reg = lin_reg.intercept_ + lin_reg.coef_[0] * x_plot
line_lin_reg_2 = lin_reg_2.intercept_ + lin_reg_2.coef_[0] * x_plot
line_lin_reg_3 = lin_reg_3.intercept_ + lin_reg_3.coef_[0] * x_plot

In [ ]:
plot_models(
    {
        "Linear model 1": line_lin_reg,
        "Linear model 2": line_lin_reg_2,
        "Linear model 3": line_lin_reg_3,
    },
    "Comparison of linear models using different training data",
    points="all",
)

The three linear models are not identical, but their intercepts and slopes are similar.

These models are not very sensitive to changes in the training data.

Let's see how the more flexible models perform.

In [ ]:
# Degree-8 features for each split, standardised as above (one scaler per split)
polynomial_features = PolynomialFeatures(degree=8, include_bias=False)

scaler_1 = StandardScaler()
x_poly_train = scaler_1.fit_transform(polynomial_features.fit_transform(X_train))
x_poly_test = scaler_1.transform(polynomial_features.transform(X_test))

scaler_2 = StandardScaler()
x_poly_train_2 = scaler_2.fit_transform(polynomial_features.fit_transform(X_train_split2))
x_poly_test_2 = scaler_2.transform(polynomial_features.transform(X_test_split2))

scaler_3 = StandardScaler()
x_poly_train_3 = scaler_3.fit_transform(polynomial_features.fit_transform(X_train_split3))
x_poly_test_3 = scaler_3.transform(polynomial_features.transform(X_test_split3))

# Flexible model 1
poly_reg = LinearRegression()
poly_reg.fit(x_poly_train, y_train)

# Flexible model 2
poly_reg_2 = LinearRegression()
poly_reg_2.fit(x_poly_train_2, y_train_split2)

# Flexible model 3
poly_reg_3 = LinearRegression()
poly_reg_3.fit(x_poly_train_3, y_train_split3)

In [ ]:
# Transform the plotting grid the same way as each model's training data
x_plot_poly = polynomial_features.transform(x_plot.to_frame())
line_poly_1 = poly_reg.predict(scaler_1.transform(x_plot_poly))
line_poly_2 = poly_reg_2.predict(scaler_2.transform(x_plot_poly))
line_poly_3 = poly_reg_3.predict(scaler_3.transform(x_plot_poly))

In [ ]:
plot_models(
    {
        "Polynomial model 1": line_poly_1,
        "Polynomial model 2": line_poly_2,
        "Polynomial model 3": line_poly_3,
    },
    "Comparison of highly flexible models using different training data",
    points="all",
)

Again, even though these flexible models fit the training data very well, they do not generalise well: they will not predict well on unseen data.

These flexible models change a lot depending on the training data: they are **high-variance** models.

To understand how these curves came about, you can plot the training data of each curve separately from the test data.



### Try It: The Middle Ground

The straight lines were steady but wrong (high bias), and the degree-8 curves changed a lot from split to split (high variance). What about a model in between? Train a **degree-3** model on each of the three splits, plot the three curves with `plot_models()`, and calculate each model's test RMSE. Does degree 3 behave like the straight lines or like the degree-8 curves?

In [ ]:
splits = [
    (X_train, X_test, y_train, y_test),
    (X_train_split2, X_test_split2, y_train_split2, y_test_split2),
    (X_train_split3, X_test_split3, y_train_split3, y_test_split3),
]
features = PolynomialFeatures(degree=3, include_bias=False)
lines = {}
for k, (X_tr, X_te, y_tr, y_te) in enumerate(splits, start=1):
    # TODO: fit a LinearRegression on the degree-3 features of X_tr
    model = ...
    # TODO: predict along the plotting grid, x_plot.to_frame(), for the plot
    lines[f"Degree 3 model {k}"] = ...
    # TODO: calculate the test RMSE on X_te and y_te
    rmse = ...
    print(f"split {k}: test RMSE {rmse:.2f}")

plot_models(lines, "Degree 3 models using different training data", points="all")

<details><summary>Show answer</summary>

```python
splits = [
    (X_train, X_test, y_train, y_test),
    (X_train_split2, X_test_split2, y_train_split2, y_test_split2),
    (X_train_split3, X_test_split3, y_train_split3, y_test_split3),
]
features = PolynomialFeatures(degree=3, include_bias=False)
lines = {}
for k, (X_tr, X_te, y_tr, y_te) in enumerate(splits, start=1):
    model = LinearRegression().fit(features.fit_transform(X_tr), y_tr)
    lines[f"Degree 3 model {k}"] = model.predict(features.transform(x_plot.to_frame()))
    rmse = root_mean_squared_error(y_te, model.predict(features.transform(X_te)))
    print(f"split {k}: test RMSE {rmse:.2f}")

plot_models(lines, "Degree 3 models using different training data", points="all")
```

The three degree-3 curves lie almost on top of each other and on the true relationship. Their test RMSEs are 19.34, 19.10 and 17.12:

| Model | Test RMSE on the three splits | Spread (standard deviation) |
|---|---|---|
| Straight line | 43.73, 43.42, 42.36 | 0.59 |
| Degree 3 | 19.34, 19.10, 17.12 | 0.99 |
| Degree 8 | 32.92, 100.38, 170.49 | 56.17 |

Degree 3 is almost as **steady** as the straight line (low variance) and far more **accurate** (low bias). That is the goal of the bias-variance trade-off: not the simplest or the most flexible model, but the one whose flexibility matches the real relationship.

</details>

## Summary



In this notebook you:

- Split the data into a training and a test set, and calculated the RMSE on each.
- Trained linear models with polynomial features of degree 1, 2, 3 and 8.
- Compared training and test RMSE: the straight line **underfits** (high bias), the degree-3 model matches the true relationship best, and the degree-8 model starts to **overfit** (high variance), fitting noise in the training data.
- Compared models trained on three different splits: the straight lines barely change, while the degree-8 curves change a lot.

**Bias** is the error from the simplifying assumptions a model makes. **Variance** is how much the model changes when it is trained on different data. The **bias-variance trade-off** says that a slightly biased model can be the better choice when it has a much lower variance. Notebook 02 shows a way to control this trade-off directly: regularisation.

## Check Your Understanding



1. Calculate the RMSE (for train and test data) of the last 6 models (3 linear regression and 3 polynomial regression (order 8)). What do you expect?

<details>
<summary>Show answer</summary>

**Expectation (bias-variance trade-off):** the linear models are not very flexible, so they are likely biased: a fairly high RMSE on both train and test data, but a low variance in the test RMSE across different train-test splits. The degree-8 polynomial models are very flexible, so they fit the training data very well (low train RMSE) but can pick up noise as if it were signal, giving larger test RMSEs and a high variance in test RMSE across different splits.

You can confirm this by predicting with the six fitted models and computing the RMSE for each:

```python
# predictions on each model's own test set
y_pred = lin_reg.predict(X_test)
y_pred_2 = lin_reg_2.predict(X_test_split2)
y_pred_3 = lin_reg_3.predict(X_test_split3)
y_poly_pred = poly_reg.predict(x_poly_test)
y_poly_pred_2 = poly_reg_2.predict(x_poly_test_2)
y_poly_pred_3 = poly_reg_3.predict(x_poly_test_3)

# predictions on each model's own train set
y_pred_train = lin_reg.predict(X_train)
y_pred_2_train = lin_reg_2.predict(X_train_split2)
y_pred_3_train = lin_reg_3.predict(X_train_split3)
y_poly_pred_train = poly_reg.predict(x_poly_train)
y_poly_pred_2_train = poly_reg_2.predict(x_poly_train_2)
y_poly_pred_3_train = poly_reg_3.predict(x_poly_train_3)

results = pd.DataFrame({"model": ["linear 1", "linear 2", "linear 3", "8-degree 1", "8-degree 2", "8-degree 3"]})

y_test_preds = (y_pred, y_pred_2, y_pred_3, y_poly_pred, y_poly_pred_2, y_poly_pred_3)
y_test_sets = (y_test, y_test_split2, y_test_split3, y_test, y_test_split2, y_test_split3)
y_train_preds = (y_pred_train, y_pred_2_train, y_pred_3_train, y_poly_pred_train, y_poly_pred_2_train, y_poly_pred_3_train)
y_train_sets = (y_train, y_train_split2, y_train_split3, y_train, y_train_split2, y_train_split3)

for idx in range(6):
    results.loc[idx, "train_RMSE"] = root_mean_squared_error(y_train_sets[idx], y_train_preds[idx])
    results.loc[idx, "test_RMSE"] = root_mean_squared_error(y_test_sets[idx], y_test_preds[idx])
results
```

**What you see:**

| Model | Train RMSE | Test RMSE |
|---|---|---|
| linear 1, 2, 3 | 33.08, 37.45, 44.14 | 43.73, 43.42, 42.36 |
| 8-degree 1, 2, 3 | 13.80, 13.08, 18.54 | 32.92, 100.38, 170.49 |

The linear models have a higher training error, but their test RMSE barely changes across the splits (standard deviation 0.59): the typical low-variance, biased model. The degree-8 models fit their training data much better, but their test RMSE swings from 33 to 170 depending on the split (standard deviation 56.17), exactly the high-variance behaviour the trade-off predicts. This is why a biased but stable model is often trusted over a high-variance one: you at least know the error will be similar each time.
</details>

2. Train (8) polynomial models with different degrees of features (polynomial degree from 1-8). Plot the train RMSE and the test RMSE of the models against model flexibility (polynomial degree 1-8), like the very first plot (from Wikipedia) in this notebook. What trend do you see in the train RMSE and the test RMSE, and what explains these trends?

<details>
<summary>Show answer</summary>

Fit a linear model on the first `i` polynomial features for `i` from 1 to 8, recording the train and test RMSE at each step, then plot both against complexity:

```python
rmse_train, rmse_test = [], []
for i in range(1, 9):
    model = LinearRegression().fit(x_poly_train[:, :i], y_train)
    rmse_train.append(root_mean_squared_error(model.predict(x_poly_train[:, :i]), y_train))
    rmse_test.append(root_mean_squared_error(model.predict(x_poly_test[:, :i]), y_test))

plt.plot(range(1, 9), rmse_train, marker="o", label="train RMSE")
plt.plot(range(1, 9), rmse_test, marker="x", label="test RMSE")
plt.xlabel("Model complexity")
plt.ylabel("RMSE")
plt.legend()
plt.show()
```

**Trend:** the train RMSE keeps falling as complexity grows, because a more flexible model fits the training data ever more closely. The test RMSE is high for very simple models (degree 1-2, underfitting / high bias), reaches its best values for moderate complexity (19.34 at degree 3, and 19.07 at degree 5, the lowest), then rises again, sharply at degree 8 (32.92: overfitting / high variance). Once the model is more flexible than the true relationship between x and y, it starts treating noise as signal, so it generalises worse on unseen data.
</details>

3. After trying out these different models, which model would you choose, and why?

<details>
<summary>Show answer</summary>

Choose the least complex model that still performs well on the test data: here degree 3. Its test RMSE (19.34) is almost as good as the best one, degree 5 (19.07), and far better than degree 8. The simpler model generalises about as well while being more stable and easier to interpret, so there is no reason to pay for the extra flexibility. This is Occam's razor applied to modelling: among models that perform about equally well, prefer the simplest.
</details>

4. What can you do to prevent overfitting?

<details>
<summary>Show answer</summary>

A few options:

- Collect more training data, so the model has less room to memorise noise.
- Use a less flexible model (here, a lower polynomial degree).
- Add regularisation, which penalises large coefficients. For linear regression that means L1 (Lasso) or L2 (Ridge), which is exactly the topic of the next notebook.
</details>


## References & Further Reading

- [**Underfitting vs. Overfitting (scikit-learn)**](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html): the same polynomial experiment, with cross-validation.
- [**PolynomialFeatures (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html): the transformer used to build the flexible models.
- [**An Introduction to Statistical Learning**](https://www.statlearning.com/): a free textbook; section 2.2 explains the bias-variance trade-off with similar plots.